# 01 — Data Collection

**資料範圍：2018–2024 球季 · 僅 Statcast（不使用 FanGraphs 打擊表）**

**流程：**
1. 逐年讀取或下載 `statcast_YYYY.parquet`（`REFRESH_BATTING=False` 時沿用既有檔）。
2. 合併後依 **`(batter, season)`** 彙總打席統計（見下方公式）。
3. 輸出 **`data/raw/batting_by_season_2018_2024.parquet`**（逐季球員列）；並可由 `scripts/build_pipeline_outputs.py` 重現 **`projection_panel.parquet`**。
4. 個別球員熱區圖用 CSV 仍從 `statcast_2024.parquet` 篩選。

| 欄位 | 計算方式 |
|---|---|
| AVG / OBP / SLG / ISO / BABIP | 事件欄位（`events`）逐打席統計 |
| BIP% | 場內擊球結束之事件數（見 `bb_pipeline.statcast_season.BIP_EVENTS`）÷ PA |
| wOBA | Statcast `woba_value` / `woba_denom`（逐季） |
| wRC+ | 各季聯盟加權平均 wOBA 代入 `(wOBA - lg_wOBA) / wOBA_scale / lg_R_PA + 1) × 100` |
| age_bat_median | 該季 PA 列 `age_bat` 之中位數 |

多年「領先榜」型 CSV 已不再於此產生；分析改以逐季面板為主。

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pybaseball import cache, playerid_lookup, playerid_reverse_lookup

cache.enable()

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1-2  Statcast 逐年 → **逐季打者面板**

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from pandas.errors import ParserError
from pybaseball import cache, statcast

from bb_pipeline.position_lahman import attach_primary_position
from bb_pipeline.projection_dataset import build_projection_panel
from bb_pipeline.statcast_season import aggregate_batting_by_season

cache.enable()

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2018
END_YEAR = 2024
REFRESH_BATTING = False
SEASON_OUT = RAW_DIR / "batting_by_season_2018_2024.parquet"


def _season_statcast_range(year: int) -> tuple[str, str]:
    return f"{year}-03-20", f"{year}-10-15"


def _fetch_statcast_year(year: int) -> pd.DataFrame:
    legacy_2024 = RAW_DIR / "statcast_2024.parquet"
    year_pq = RAW_DIR / f"statcast_{year}.parquet"
    if year == END_YEAR and legacy_2024.exists() and not REFRESH_BATTING:
        print(f"  {year}: read {legacy_2024.name}")
        return pd.read_parquet(legacy_2024)
    if year_pq.exists() and not REFRESH_BATTING:
        print(f"  {year}: read {year_pq.name}")
        return pd.read_parquet(year_pq)

    st, en = _season_statcast_range(year)
    print(f"  {year}: Statcast download {st} … {en}")
    try:
        sc_y = statcast(st, en)
    except ParserError:
        print("       full-season parse failed; monthly chunks…")
        parts = []
        windows = [
            (f"{year}-03-20", f"{year}-04-30"),
            (f"{year}-05-01", f"{year}-05-31"),
            (f"{year}-06-01", f"{year}-06-30"),
            (f"{year}-07-01", f"{year}-07-31"),
            (f"{year}-08-01", f"{year}-08-31"),
            (f"{year}-09-01", f"{year}-09-30"),
            (f"{year}-10-01", f"{year}-10-15"),
        ]
        for a, b in windows:
            print(f"         {a} … {b}")
            parts.append(statcast(a, b))
        sc_y = pd.concat(parts, ignore_index=True)

    sc_y.to_parquet(year_pq, index=False)
    print(f"       saved {len(sc_y):,} rows → {year_pq.name}")
    return sc_y


frames = [_fetch_statcast_year(y) for y in range(START_YEAR, END_YEAR + 1)]
sc_all = pd.concat(frames, ignore_index=True)

batting_season = aggregate_batting_by_season(sc_all)
batting_season.to_parquet(SEASON_OUT, index=False)
print(f"Season panel: {batting_season.shape} → {SEASON_OUT.name}")

batting_season = attach_primary_position(batting_season)
proj = build_projection_panel(batting_season, min_pa_per_feature_season=100)
proj_path = PROC_DIR / "projection_panel.parquet"
proj.to_parquet(proj_path, index=False)
print(f"Projection rows: {proj.shape} → {proj_path.name}")

batting_season.head()


## 1-3  個別球員 Statcast 逐球資料（熱區圖用）

In [3]:
PARQUET_PATH = RAW_DIR / 'statcast_2024.parquet'
players = [
    ('judge',   'aaron',  592450),
    ('betts',   'mookie', 605141),
    ('acuna',   'ronald', 660670),
]

sc_full = pd.read_parquet(PARQUET_PATH)
print(f'全聯盟 Statcast 總筆數：{len(sc_full):,}')

for last, first, mlbam_id in players:
    out_path = RAW_DIR / f'statcast_{first}_{last}_2024.csv'
    if out_path.exists():
        n = len(pd.read_csv(out_path))
        print(f'{first} {last} (ID={mlbam_id})：{n} 筆（已快取）')
        continue
    df_p = sc_full[sc_full['batter'] == mlbam_id].copy()
    df_p.to_csv(out_path, index=False)
    print(f'{first} {last} (ID={mlbam_id})：{len(df_p)} 筆 → {out_path.name}')

全聯盟 Statcast 總筆數：695,264
aaron judge (ID=592450)：2880 筆（已快取）
mookie betts (ID=605141)：1932 筆（已快取）
ronald acuna (ID=660670)：852 筆（已快取）


## 確認輸出

In [4]:
import glob
files = sorted(glob.glob(str(RAW_DIR / '*.csv')) + glob.glob(str(RAW_DIR / '*.parquet')))
for f in files:
    path = Path(f)
    size_mb = path.stat().st_size / 1e6
    print(f'{path.name:50s}  {size_mb:.1f} MB')

batting_stats_2018_2024.csv                         0.1 MB
statcast_2018.parquet                               90.0 MB
statcast_2019.parquet                               91.7 MB
statcast_2020.parquet                               35.7 MB
statcast_2021.parquet                               90.5 MB
statcast_2022.parquet                               91.1 MB
statcast_2023.parquet                               98.0 MB
statcast_2024.parquet                               102.1 MB
statcast_aaron_judge_2024.csv                       1.8 MB
statcast_mookie_betts_2024.csv                      1.2 MB
statcast_ronald_acuna_2024.csv                      0.5 MB
